# Context-Aware Revenue Optimization in Food Delivery Platforms Using Deep Learning and Stochastic Demand Simulation

*by Abdulrahman Jaber Ageeli, June 2026*

## Introduction

Food delivery platforms operate in highly dynamic environments where customer demand varies according to contextual variables such as delivery distance, time of day, and environmental conditions. Static pricing policies fail to capture this variability and often lead to suboptimal revenue outcomes.

This project proposes a context-aware revenue optimization framework that combines stochastic demand simulation with supervised machine learning. Rather than relying solely on historical binary outcomes, we construct a controlled economic simulation of customer acceptance behavior. This allows us to embed nonlinear price sensitivity and contextual interactions explicitly.

Two models are compared:

- Logistic Regression (linear baseline)
- Deep Neural Network (nonlinear model)

The goal is to estimate acceptance probability under counterfactual pricing scenarios and determine the revenue-maximizing delivery price.

## Methods

### Data and Feature Engineering

The base dataset contains operational information from a food delivery platform, including order creation time and estimated driving duration.

Distance is engineered from driving duration:

$$
distance = \frac{duration\ (seconds)}{3600} \times 40
$$

where 40 km/h is assumed as the average city speed.

Peak periods are defined as:

- 11:00–14:00 (Lunch)
- 18:00–21:00 (Dinner)

Weather is simulated conditionally:

$$
P(rain) = 0.2 + 0.1 \times is\_peak
$$

A dynamic base delivery price is defined as:

$$
P_{base} = 8 + 0.8 \times distance
$$

To prevent data leakage, the dataset is split into training and testing sets before price expansion.

### Stochastic Demand Simulation

For each order context, multiple counterfactual prices are generated:

$$
price = P_{base} + \delta, \quad \delta \sim U(-0.5P_{base}, +0.5P_{base})
$$

Price sensitivity is modeled as a nonlinear function of context:

$$
\alpha(X) =
0.15
+ 0.05 \cdot distance
+ 0.02 \cdot distance^2
+ 0.08 \cdot (distance \times is\_rainy)
- 0.07 \cdot is\_peak
$$

Acceptance probability follows a logistic model with noise:

$$
z = -\alpha(X)(price - P_{base}) + \epsilon
$$

$$
P(accept) = \sigma(z)
$$

where $\epsilon \sim N(0, 0.5)$.

This formulation creates a nonlinear demand surface that justifies the use of deep learning.

### Modeling

Logistic Regression is used as a linear baseline model.

The Deep Neural Network consists of:

- Dense(64) → ReLU
- Dropout(0.2)
- Dense(32) → ReLU
- Dense(16) → ReLU
- Output layer with sigmoid activation

Early stopping is applied to reduce overfitting.

Performance is evaluated using:

- ROC-AUC (discrimination)
- Brier Score (calibration)
- Revenue optimization curve

## Results

The Deep Neural Network consistently outperformed Logistic Regression in both discrimination and calibration.

The DNN achieved higher ROC-AUC and lower Brier Score, confirming that nonlinear contextual interactions provide additional predictive signal.

The ROC curve demonstrates strong separation between accepted and rejected orders, with the DNN slightly dominating the linear baseline.

Using the trained DNN, expected revenue was computed as:

$$
Revenue = price \times P(accept)
$$

The model identifies a revenue-maximizing price that balances margin and acceptance probability, demonstrating practical applicability to dynamic pricing systems.

## Conclusions

This project demonstrates that combining stochastic economic modeling with supervised deep learning provides a structured approach to dynamic pricing.

Key findings:

1. Preventing data leakage before counterfactual expansion is essential for valid evaluation.
2. Nonlinear contextual interactions justify deep neural networks.
3. Deep learning improves both predictive accuracy and revenue optimization.

The framework can be extended with real weather, geographic, and traffic data for deployment in production pricing systems.

### References

* Goodfellow, I., Bengio, Y., & Courville, A. (2016). Deep Learning. MIT Press.
* Bishop, C. M. (2006). Pattern Recognition and Machine Learning. Springer.
* Friedman, J., Hastie, T., & Tibshirani, R. (2001). The Elements of Statistical Learning. Springer.
* McFadden, D. (1974). Conditional Logit Analysis of Qualitative Choice Behavior.
* Kaggle DoorDash Dataset.